# Batch Document Processing

Process multiple documents in batch using template-based OCR pipeline.

In [1]:
from pathlib import Path
import sys

# Find project root by looking for src/ directory
project_root = Path.cwd()
while not (project_root / 'src').exists() and project_root.parent != project_root:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

# Define path constants
DATA_DIR = project_root / 'data'
TEMPLATES_DIR = DATA_DIR / 'templates'
INPUTS_DIR = DATA_DIR / 'inputs'
OUTPUTS_DIR = project_root / 'outputs'
PROCESSED_DIR = OUTPUTS_DIR / 'processed'

import json
import cv2
from typing import Dict, List, Tuple

from src.alignment.phase_1_alignment import FeatureBasedAligner, AlignmentConfig
from src.visualization.diff_visualization import create_red_overlay_diff
from src.ocr.debug_utils import debug_slice_ocr, build_paddle_ocr_debugger
from src.template import (
    load_zones_from_template,
    build_union_mask,
    apply_mask,
    extract_polygon_roi,
    detect_checkboxes_from_template_image,
    ROI_PADDING,
    DET_DB_THRESH,
    DET_DB_BOX_THRESH,
    DET_DB_UNCLIP_RATIO,
    DET_LIMIT_SIDE_LEN,
    DET_LIMIT_TYPE,
    USE_DOC_ORIENTATION_CLASSIFY,
    USE_DOC_UNWARPING,
    USE_TEXTLINE_ORIENTATION,
)
from src.ocr.ocr_extraction import FastOCRExtractor, visualize_extractions, extract_roi, preprocess_roi

/home/lex/GitHub/ai-adoption-research-and-development/Template-alignment/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


## Configuration

In [2]:
DEFAULT_TEMPLATE_IMAGE = TEMPLATES_DIR / "template.jpg"
DEFAULT_TEMPLATE_JSON = TEMPLATES_DIR / "template.json"

# Notebook-style configuration
INPUT_DIR = INPUTS_DIR / "sample_forms"        # change to your folder of inputs
OUTPUT_DIR = PROCESSED_DIR                     # where json_data/debug_data will be written
TEMPLATE_IMAGE = DEFAULT_TEMPLATE_IMAGE       # or Path("/path/to/template.jpg")
TEMPLATE_JSON = DEFAULT_TEMPLATE_JSON          # or Path("/path/to/template.json")
EXTENSIONS = ("jpg", "jpeg", "png")            # tuple of extensions to process

## Helper Functions

In [3]:
def _safe_name(name: str) -> str:
    return name.replace("/", "_").replace("\\", "_")


def _load_template_assets(template_image_path: Path, template_json_path: Path) -> Tuple[Dict, Dict, Dict]:
    template_img = cv2.imread(str(template_image_path))
    if template_img is None:
        raise FileNotFoundError(f"Failed to read template image at '{template_image_path}'")
    polygon_zones, rectangle_zones, checkbox_boxes = load_zones_from_template(str(template_json_path))
    return template_img, polygon_zones, rectangle_zones, checkbox_boxes


def _build_aligner() -> FeatureBasedAligner:
    config = AlignmentConfig(
        feature_detector="ORB",
        max_features=5000,
        ratio_test_threshold=0.7,
        ransac_threshold=5.0,
        min_matches=10,
        verbose=True,
    )
    return FeatureBasedAligner(config)


def _build_extractor() -> Tuple[FastOCRExtractor, object]:
    extractor = FastOCRExtractor(
        lang="en",
        verbose=True,
        use_doc_orientation_classify=USE_DOC_ORIENTATION_CLASSIFY,
        use_doc_unwarping=USE_DOC_UNWARPING,
        use_textline_orientation=USE_TEXTLINE_ORIENTATION,
        det_db_thresh=DET_DB_THRESH,
        det_db_box_thresh=DET_DB_BOX_THRESH,
        det_db_unclip_ratio=DET_DB_UNCLIP_RATIO,
        det_limit_side_len=DET_LIMIT_SIDE_LEN,
        det_limit_type=DET_LIMIT_TYPE,
    )
    slice_debugger = build_paddle_ocr_debugger(
        lang="en",
        det_limit_side_len=DET_LIMIT_SIDE_LEN,
        det_limit_type=DET_LIMIT_TYPE,
        det_db_thresh=DET_DB_THRESH,
        det_db_box_thresh=DET_DB_BOX_THRESH,
        det_db_unclip_ratio=DET_DB_UNCLIP_RATIO,
        use_doc_orientation_classify=USE_DOC_ORIENTATION_CLASSIFY,
        use_doc_unwarping=USE_DOC_UNWARPING,
        use_textline_orientation=USE_TEXTLINE_ORIENTATION,
    )
    return extractor, slice_debugger

## Process Single Document

In [4]:
def process_single_document(
    input_image_path: Path,
    template_img,
    polygon_zones: List[Dict],
    rectangle_zones: List[Dict],
    checkbox_boxes: List[Dict],
    aligner: FeatureBasedAligner,
    extractor: FastOCRExtractor,
    slice_debugger,
    output_json_path: Path,
    debug_dir: Path,
    roi_padding: int = ROI_PADDING,
) -> Dict:
    debug_dir.mkdir(parents=True, exist_ok=True)
    slices_dir = debug_dir / "slices"
    slices_dir.mkdir(parents=True, exist_ok=True)

    input_img = cv2.imread(str(input_image_path))
    if input_img is None:
        raise FileNotFoundError(f"Failed to read input image at '{input_image_path}'")

    result = aligner.align(input_img, template_img)

    aligned_path = debug_dir / "aligned_default_orb.jpg"
    overlay_path = debug_dir / "aligned_default_orb_red_overlay.jpg"
    masked_path = debug_dir / "aligned_masked_for_ocr.png"
    extraction_vis_path = debug_dir / "extraction_visualization.jpg"
    slice_log_path = slices_dir / "slice_debug_log.txt"

    cv2.imwrite(str(aligned_path), result.aligned_image)
    red_overlay = create_red_overlay_diff(template_img, result.aligned_image, threshold=30, alpha=0.6)
    cv2.imwrite(str(overlay_path), red_overlay)

    union_mask = build_union_mask(result.aligned_image.shape, polygon_zones, rectangle_zones)
    masked_image = apply_mask(result.aligned_image, union_mask)
    cv2.imwrite(str(masked_path), masked_image)

    checkbox_results = detect_checkboxes_from_template_image(
        image=result.aligned_image,
        checkbox_boxes=checkbox_boxes,
        checked_threshold=0.15,
        verbose=False,
    )

    extracted_data: Dict[str, Dict] = {}

    for zone in polygon_zones:
        safe_name = _safe_name(zone["name"])
        roi = extract_polygon_roi(masked_image, zone["polygon"], padding=roi_padding)
        text, confidence = extractor.extract_text(roi, preprocess=True)
        extracted_data[zone["name"]] = {
            "text": text,
            "confidence": confidence,
            "bbox": zone["bbox"] if zone.get("bbox") else None,
        }
        cv2.imwrite(str(slices_dir / f"{safe_name}_roi.jpg"), roi)
        processed_roi = preprocess_roi(roi)
        cv2.imwrite(str(slices_dir / f"{safe_name}_processed.jpg"), processed_roi)
        debug_slice_ocr(
            processed_roi,
            slice_debugger,
            slices_dir,
            f"{safe_name}_processed",
            log_path=str(slice_log_path),
        )

    for zone in rectangle_zones:
        safe_name = _safe_name(zone["name"])
        roi = extract_roi(masked_image, zone["bbox"], padding=roi_padding)
        text, confidence = extractor.extract_text(roi, preprocess=True)
        extracted_data[zone["name"]] = {
            "text": text,
            "confidence": confidence,
            "bbox": zone["bbox"],
        }
        cv2.imwrite(str(slices_dir / f"{safe_name}_roi.jpg"), roi)
        processed_roi = preprocess_roi(roi)
        cv2.imwrite(str(slices_dir / f"{safe_name}_processed.jpg"), processed_roi)
        debug_slice_ocr(
            processed_roi,
            slice_debugger,
            slices_dir,
            f"{safe_name}_processed",
            log_path=str(slice_log_path),
        )

    combined_output = {**extracted_data, **checkbox_results}

    output_json_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(combined_output, f, indent=2, ensure_ascii=False)

    visualize_extractions(result.aligned_image, extracted_data, str(extraction_vis_path))
    return combined_output

## Run Batch Processing

In [5]:
def run_batch(
    input_dir: Path,
    output_dir: Path,
    template_image_path: Path = DEFAULT_TEMPLATE_IMAGE,
    template_json_path: Path = DEFAULT_TEMPLATE_JSON,
    extensions: Tuple[str, ...] = ("jpg", "jpeg", "png"),
) -> None:
    input_dir = input_dir.expanduser().resolve()
    output_dir = output_dir.expanduser().resolve()
    output_json_dir = output_dir / "json_data"
    output_debug_dir = output_dir / "debug_data"

    template_img, polygon_zones, rectangle_zones, checkbox_boxes = _load_template_assets(
        template_image_path, template_json_path
    )
    aligner = _build_aligner()
    extractor, slice_debugger = _build_extractor()

    image_paths: List[Path] = []
    for ext in extensions:
        image_paths.extend(sorted(input_dir.glob(f"*.{ext}")))
    image_paths = sorted(set(image_paths))

    if not image_paths:
        raise FileNotFoundError(
            f"No input images found in '{input_dir}' for extensions: {', '.join(extensions)}"
        )

    print(f"[BATCH] Found {len(image_paths)} images in {input_dir}")
    successes, failures = 0, 0
    for img_path in image_paths:
        stem = img_path.stem
        json_path = output_json_dir / f"{stem}.json"
        debug_dir = output_debug_dir / stem
        try:
            print(f"[BATCH] Processing {img_path.name} -> {debug_dir}")
            process_single_document(
                input_image_path=img_path,
                template_img=template_img,
                polygon_zones=polygon_zones,
                rectangle_zones=rectangle_zones,
                checkbox_boxes=checkbox_boxes,
                aligner=aligner,
                extractor=extractor,
                slice_debugger=slice_debugger,
                output_json_path=json_path,
                debug_dir=debug_dir,
                roi_padding=ROI_PADDING,
            )
            successes += 1
        except Exception as exc:
            failures += 1
            print(f"[BATCH][ERROR] {img_path.name}: {exc}")

    print(
        f"[BATCH] Completed. Successes: {successes}, Failures: {failures}. "
        f"JSON in '{output_json_dir}', debug artifacts in '{output_debug_dir}'."
    )

## Execute Batch Processing

In [6]:
run_batch(
    input_dir=Path(INPUT_DIR),
    output_dir=Path(OUTPUT_DIR),
    template_image_path=Path(TEMPLATE_IMAGE),
    template_json_path=Path(TEMPLATE_JSON),
    extensions=tuple(EXTENSIONS),
)

[INFO] Using ORB detector (Oriented FAST and Rotated BRIEF)
       Pros: 24x faster than SIFT, free license, efficient
       Cons: Less robust to extreme rotations (>45°)
[OCR] Initializing PaddleOCR...
      Language: en


/home/lex/GitHub/ai-adoption-research-and-development/Template-alignment/.venv/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/lex/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/lex/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/lex/.paddlex/official_models/en

[OCR] Ready!


Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/lex/.paddlex/official_models/en_PP-OCRv5_mobile_rec`.


[BATCH] Found 10 images in /home/lex/GitHub/ai-adoption-research-and-development/Template-alignment/data/inputs/sample_forms
[BATCH] Processing 2025-11-13_135834.jpg -> /home/lex/GitHub/ai-adoption-research-and-development/Template-alignment/outputs/processed/debug_data/2025-11-13_135834

PHASE 1: GLOBAL FEATURE-BASED REGISTRATION

[STEP 1.1] Feature Detection and Matching
[INFO] Detected 5000 keypoints
[INFO] Detected 5000 keypoints
[INFO] Found 1673/5000 good matches (33.5%)

[STEP 1.2] Homography Computation (RANSAC)
[INFO] RANSAC found 1497/1673 inliers (89.5%)
[INFO] Mean reprojection error: 1.03 pixels

[STEP 1.3] Image Warping and Registration
[INFO] Warped input image to template size: 2610x3348

----------------------------------------------------------------------
ALIGNMENT QUALITY METRICS:
----------------------------------------------------------------------
  ✓ Inlier ratio: 89.5% (target: >40%)
  ✓ Reprojection error: 1.03px (target: <5px)
  ✓ Total matches: 1673
  ✓ Inli